# IT2011 - Artificial Intelligence and Machine Learning
## Progress Review I: Data Preprocessing & Exploratory Data Analysis (EDA)
### Group ID: `2026-Y2-S1-MET-23`
### Member 3: Fernando B. K. H. (IT25102631)
### Assigned Technique: Categorical Multi-Label Encoding (Genres Feature Engineering)

---
### 1. Technique Overview & Academic Justification
The `genres` attribute provides crucial contextual metadata about each movie review:
1. **Multi-Label Nature:** Unlike single-label categorical features (e.g., gender or nationality), each movie in our dataset belongs to **multiple genres simultaneously** (e.g., `['Action', 'Adventure', 'Sci-Fi']`).
2. **String Serialization:** In the raw CSV file, genres are formatted as string representations of Python lists (e.g., `"['Drama', 'Romance']"`), requiring safe literal evaluation.
3. **Encoding Methodology:** Standard `LabelEncoder` or single-column `OneHotEncoder` cannot handle multi-label collections. We must use `MultiLabelBinarizer` from `sklearn.preprocessing` to generate binary indicator columns for each unique genre tag.

**Viva Objective:** Demonstrate parsing of stringified lists, binary matrix encoding, and correlation between movie genres and emotions.


In [ ]:
import os
import ast
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MultiLabelBinarizer

os.makedirs('../results/eda_visualizations', exist_ok=True)
sns.set_theme(style="whitegrid", palette="muted")


### 2. Loading the Raw Dataset

In [ ]:
DATA_PATH = '../data/raw/Movies_Reviews_modified_version1.csv'
df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df):,} reviews.")
df[['movie_name', 'genres', 'emotion']].head()


### 3. Parsing & Multi-Label Binarization Pipeline

In [ ]:
def parse_genres(genre_str):
    """Safely parse stringified list into a clean list of genres."""
    if pd.isna(genre_str):
        return []
    try:
        parsed = ast.literal_eval(str(genre_str))
        if isinstance(parsed, list):
            return [str(g).strip() for g in parsed if g]
        return [str(parsed).strip()]
    except Exception:
        # Fallback regex split
        return [g.strip().replace("'", "").replace('"', "") for g in str(genre_str).strip('[]').split(',') if g.strip()]

# Apply parsing
df['parsed_genres'] = df['genres'].apply(parse_genres)

# Instantiate MultiLabelBinarizer
mlb = MultiLabelBinarizer()
genre_encoded_matrix = mlb.fit_transform(df['parsed_genres'])

# Convert to DataFrame
genre_df = pd.DataFrame(genre_encoded_matrix, columns=[f"genre_{g}" for g in mlb.classes_])
print(f"Total Unique Genres Identified: {len(mlb.classes_)}")
print("Identified Genres:", list(mlb.classes_))
genre_df.head()


### 4. Genre Frequencies & Co-occurrence Matrix

In [ ]:
# Compute total occurrences per genre
genre_counts = genre_df.sum().sort_values(ascending=False)
genre_counts.index = [idx.replace('genre_', '') for idx in genre_counts.index]

# Compute genre co-occurrence correlation matrix for top 12 genres
top_genres = genre_counts.head(12).index
top_genre_cols = [f"genre_{g}" for g in top_genres]
genre_corr = genre_df[top_genre_cols].corr()
genre_corr.columns = top_genres
genre_corr.index = top_genres


### 5. Individual EDA Visualizations (Viva Presentation Requirement)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Left Plot: Genre distribution
sns.barplot(x=genre_counts.values, y=genre_counts.index, palette='crest', ax=axes[0])
axes[0].set_title('Frequency of Unique Movie Genres Across Dataset', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Number of Movies / Reviews', fontsize=11)
axes[0].set_ylabel('Genre', fontsize=11)

# Right Plot: Genre Co-occurrence Heatmap
sns.heatmap(genre_corr, annot=True, fmt=".2f", cmap="coolwarm", cbar=True, ax=axes[1], linewidths=0.5)
axes[1].set_title('Co-occurrence Correlation Among Top 12 Genres', fontsize=13, fontweight='bold')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
output_plot_path = '../results/eda_visualizations/member3_genre_cooccurrence_heatmap.png'
plt.savefig(output_plot_path, dpi=300, bbox_inches='tight')
print(f"EDA plot saved successfully to: {output_plot_path}")
plt.show()


### 6. Key Findings & Viva Talking Points (For Fernando B. K. H.)

> **Viva Preparation Notes:**
> 1. **Why can't standard One-Hot Encoding be used for genres?**
>    Standard one-hot encoding assumes mutually exclusive categories (single label per record). Since a movie can be simultaneously *Action*, *Thriller*, and *Sci-Fi*, `MultiLabelBinarizer` produces an orthogonal binary indicator matrix where multiple columns can equal 1.
> 2. **What does the genre co-occurrence analysis show?**
>    `Drama` and `Comedy` are the most prevalent genres in the dataset. Strong positive co-occurrences exist between `Action` and `Adventure`, and `Action` and `Thriller`.
> 3. **How does this assist model prediction?**
>    Certain genres have strong statistical priors with specific emotions—for instance, *Horror* strongly correlates with `fear`, *Comedy* with `joy`, and *Drama* with `sadness`. Providing these binary features enriches downstream classifiers.
